# Module 4: Hands-On - RNN and LSTM Architectures

## Objectives:
## Objectives:
1. Understand the architecture of RNN and LSTM networks.
2. Explore the outputs of RNN hidden states and LSTM gate activations.
3. Compare RNN and LSTM on handling sequential dependencies.

Tokenization and Preprocessing

In [ ]:
# Sample SMS message from SMS Spam dataset
sms_message = "Free entry in a weekly competition to win FA Cup final tickets. Text FA to 12345."

# Simple preprocessing: Tokenization and integer encoding
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer()
tokenizer.fit_on_texts([sms_message])
sequence = tokenizer.texts_to_sequences([sms_message])[0]

print("Original SMS Message:", sms_message)
print("Tokenized Sequence:", sequence)

RNN Hidden State Output

In [ ]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Embedding

# Parameters
timesteps = len(sequence)
features = 1

# Prepare data
x_data = np.expand_dims(sequence, axis=0)  # Batch size = 1
x_data = np.expand_dims(x_data, axis=-1)   # Add feature dimension

# Build RNN model
model = Sequential([
    SimpleRNN(8, activation='tanh', input_shape=(timesteps, features), return_sequences=True)
])

# Initialize model
model.compile(optimizer='adam', loss='mse')

# Get RNN outputs
rnn_output = model.predict(x_data)
print("RNN Hidden States Shape:", rnn_output.shape)
print("RNN Hidden States:", rnn_output[0])  # Outputs for each timestep

LSTM Gate Outputs

In [16]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import LSTM, Input, RNN
from tensorflow.keras.models import Model

# Custom LSTM cell to extract intermediate gate outputs
class LSTMWithGateOutputs(tf.keras.layers.LSTMCell):
    def call(self, inputs, states, training=None):
        # Get the internal gate computations
        h_tm1 = states[0]  # previous memory state
        c_tm1 = states[1]  # previous carry state

        # Extract the gate computations from the LSTMCell's internal method
        z = tf.keras.backend.dot(inputs, self.kernel)
        z += tf.keras.backend.dot(h_tm1, self.recurrent_kernel)
        z = tf.keras.backend.bias_add(z, self.bias)

        z0, z1, z2, z3 = tf.split(z, 4, axis=1)

        i = self.recurrent_activation(z0)
        f = self.recurrent_activation(z1)
        c = self.activation(z2)
        o = self.recurrent_activation(z3)

        # Calculate new cell state and output
        c = f * c_tm1 + i * c
        h = o * self.activation(c)

        # Return the output and the new states, along with the gate outputs
        return [h, f, i, o], [h, c]

# Define input sequence
timesteps = len(sequence)
features = 1

x_data = np.expand_dims(sequence, axis=0)  # Batch size = 1
x_data = np.expand_dims(x_data, axis=-1)  # Add feature dimension

# Create input and LSTM layer with custom outputs
input_layer = Input(shape=(timesteps, features))
# Initialize the LSTM cell without return_sequences and return_state
lstm_cell = LSTMWithGateOutputs(units=8, activation='tanh')
# Create an RNN layer using the custom cell, and specify return_sequences and return_state here
rnn_layer = tf.keras.layers.RNN(
    lstm_cell, return_sequences=True, return_state=True
)
# Get outputs from the RNN layer
outputs = rnn_layer(input_layer)

# Unpack the outputs - hidden states, ft, it, ot, final_h, final_c
lstm_layer = outputs[0]  # Hidden states for each timestep
final_h = outputs[1]    # Final hidden state
final_c = outputs[2]    # Final cell state

# The intermediate gate outputs (ft, it, ot) are now directly available
ft = outputs[0][:, :, 0:8]  # Assuming 8 units, the gate outputs are in the sequence
it = outputs[0][:, :, 8:16] # Assuming 8 units, the gate outputs are in the sequence
ot = outputs[0][:, :, 16:24] # Assuming 8 units, the gate outputs are in the sequence

# Build model
model = Model(inputs=input_layer, outputs=[lstm_layer, ft, it, ot, final_h, final_c])
model.compile(optimizer='adam', loss='mse')

# Get outputs
lstm_hidden_states, ft_output, it_output, ot_output, final_hidden, final_cell = model.predict(x_data)

# Display results
print("Forget Gate Outputs (ft):", ft_output)
print("\nInput Gate Outputs (it):", it_output)
print("\nOutput Gate Outputs (ot):", ot_output)
print("\nFinal Hidden State:", final_hidden)
print("\nFinal Cell State:", final_cell)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 508ms/step
Forget Gate Outputs (ft): [[[[-0.17864339  0.03161447  0.2907662   0.30067432  0.08525003
     0.00956654 -0.1032269  -0.18918619]
   [-0.2573812   0.06565987  0.46289748  0.5410501   0.1751056
     0.00656452 -0.19166774 -0.31620142]
   [-0.2705941   0.10660123  0.5459538   0.69439524  0.262645
     0.00279067 -0.26418677 -0.38423872]
   [-0.25556493  0.15502423  0.578348    0.78793275  0.35008198
     0.00139    -0.32107213 -0.40853015]
   [-0.23246498  0.21112551  0.5839542   0.84616137  0.43701568
     0.00197111 -0.3634737  -0.40876055]
   [-0.20963039  0.27454028  0.57757145  0.88345087  0.5179009
     0.00315946 -0.39341906 -0.39962476]
   [-0.35258663  0.1517078   0.32958394  0.51902354  0.2896816
    -0.02168183 -0.1857726  -0.40294576]
   [-0.1959093   0.30535832  0.54677314  0.8910943   0.49558875
     0.00210204 -0.38061392 -0.40420157]]]


 [[[ 0.4968803   0.7596396   0.5438666   0.68991745  0.84929395
     0.52464306  0.8424223   0.9

## Reflection Questions:
1. How do RNN and LSTM architectures differ in handling long-term dependencies?
2. What insights can you gain from visualizing hidden states and gate activations?
3. Why might LSTMs be better suited for tasks with longer sequences? Provide examples.